# EffectiveInference — пайплайн дистилляции (Colab, без судьи)

Прогоняет: **быстрый SFT → merge → генерация (замер tok/s) → ручной спот-чек**.

Валидация качества — самой тестирующей системой (4 сабмита/сутки). Тут только
обучаем модель и глазами проверяем, что ответы похожи на эталон.

Перед запуском: Runtime → Change runtime type → GPU (T4 бесплатно, A100/L4 быстрее).

In [ ]:
# 1. Зависимости
!pip install -q -U vllm==0.11.0 transformers==4.56.1 peft accelerate datasets bitsandbytes
# Колабовский torchao 0.10 несовместим с peft (is_torchao_available() кидает ImportError).
# torchao нам не нужен (квантизации нет) — удаляем, чтобы peft его пропускал.
!pip uninstall -y torchao

In [ ]:
# 2. Смонтировать Drive (путь и переход в папку — в ячейке 3 «Конфиг»).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Конфиг прогона — ОБЯЗАТЕЛЬНО запусти ДО шагов ниже.
# Кладём в os.environ, чтобы $VAR надёжно раскрывался в !-командах (subshell).
import os
PROJECT = '/content/drive/MyDrive/effinf_dev'   # папка с кодом и data/
os.environ['PROJECT']    = PROJECT
os.environ['BASE_MODEL'] = 'Qwen/Qwen3-1.7B'
os.environ['VARIANT']    = 'minimal'            # 'minimal' | 'rich'
os.environ['DRIVE']      = '/content/drive/MyDrive/effinf'  # куда писать lora/merged
os.environ['DTYPE']      = 'float16'            # T4. A100/L4 → 'bfloat16' или 'auto'
os.makedirs(os.environ['DRIVE'], exist_ok=True)
%cd $PROJECT
!ls data/

## Шаг 1. Быстрый SFT (T4 ~30-50 мин)

In [ ]:
%cd $PROJECT
!python train_lora.py --variant "$VARIANT" \
    --train_jsonl "data/train_${VARIANT}.jsonl" \
    --output_dir "$DRIVE/lora_$VARIANT" \
    --base_model "$BASE_MODEL" \
    --max_samples 3000 --epochs 1
# полный прогон: убрать --max_samples, --epochs 2 (лучше на A100/L4)

## Шаг 2. Merge адаптера → safetensors

In [ ]:
%cd $PROJECT
!python merge_lora.py --base_model "$BASE_MODEL" \
    --adapter "$DRIVE/lora_$VARIANT" \
    --out "$DRIVE/merged_$VARIANT"

## Шаг 3. Генерация на held-out + замер tok/s и доли упёршихся в кэп

In [ ]:
%cd $PROJECT
!python gen_candidates.py --model_dir "$DRIVE/merged_$VARIANT" --variant "$VARIANT" \
    --out "data/cand_$VARIANT.jsonl" --dtype "$DTYPE"

## Шаг 4. Ручной спот-чек — глазами сравнить ответы с эталоном (без API)

In [ ]:
%cd $PROJECT
!python peek.py --candidates "data/cand_$VARIANT.jsonl" --n 6

**Что смотреть на шаге 3-4:**
- `tok/s` и экстраполяция на 4000 запросов — влезаем ли в 15 мин (достоверно только на L4).
- `% упёршихся в max_tokens` — если высоко, модель не ставит EOS вовремя (плохо для времени).
- спот-чек: появился ли формат эталона (шаги → LaTeX → выделенный ответ), нет ли мусора/обрывов.

Если выглядит разумно — забираем `merged_$VARIANT` в посылку (см. инструкцию в чате)
и отправляем на тестирующую систему. Балл с неё = наша валидация.